## Imports

Important imports and packages used

In [ ]:
!pip install fairlearn
# from google.colab import userdata
import os
import kagglehub
import pandas as pd
import os
import csv
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, auc
import torch
import transformers
import torchvision.transforms as T
from transformers import ViTForImageClassification, ViTImageProcessor, AutoConfig
from torch.utils.data import Dataset, DataLoader
from transformers import Trainer, TrainingArguments
from transformers import AutoModelForImageClassification, AutoImageProcessor

# import selectkbest

IMG_SIZE   = (128, 128)
# AUTOTUNE   = tf.data.AUTOTUNE

label_map = {'Benign': 0,
             'Malignant': 1}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 66.5 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


## Load Dataset

In [ ]:
# os.environ["KAGGLE_USERNAME"] =  userdata.get('KAGGLE_USERNAME')
# os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

os.environ["KAGGLE_USERNAME"] = "josephgalicinao"
os.environ["KAGGLE_KEY"] = "d222ce164adf121a586cdc25319e3bbc"

ISIC 2018

In [ ]:
isic2018_path = kagglehub.dataset_download("josephgalicinao/isic-2018-dataset")

print("Path to dataset files:", isic2018_path)

100%|██████████| 237M/237M [00:02<00:00, 113MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/josephgalicinao/isic-2018-dataset/versions/1


ISIC 2018 Test

In [ ]:
# Download latest version
isic2018_test_path = kagglehub.dataset_download("josephgalicinao/isic-2018-test-dataset")

print("Path to dataset files:", isic2018_test_path)

100%|██████████| 37.3M/37.3M [00:03<00:00, 11.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/josephgalicinao/isic-2018-test-dataset/versions/1


DDI + Milk10K + ISIC Archive

In [ ]:
# Download latest version
skin_tone_path = kagglehub.dataset_download("josephgalicinao/skin-tone-dataset")

print("Path to dataset files:", skin_tone_path)

100%|██████████| 8.28G/8.28G [03:57<00:00, 37.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/josephgalicinao/skin-tone-dataset/versions/2


## Create Datasets

Create Fitzpatrick dataset + Unlabeled Fitzpatrick

In [ ]:
def get_fitzpatrick_datasets():
  # ISIC
  print("Getting ISIC...")
  df = pd.read_csv(f"{skin_tone_path}/Representative/ISIC_archive/isic_metadata.csv")
  isic_ids = df["isic_id"].values
  isic_dx = df["diagnosis_1"].map({'Malignant': 1, 'Benign': 0}).values
  isic_skin_tone = df["fitzpatrick_skin_type"].map({'I': 0, 'II': 0, 'III': 1,
                                                         'IV': 1, 'V': 2, 'VI':2}).values
  isic_patient_ids = df["patient_id"].values

  isic_paths = []
  for isic_id in isic_ids:
    isic_paths.append(f"{skin_tone_path}/Representative/ISIC_archive/{isic_id}.jpg")

  # Dataset characteristics
  # print("----- ISIC -----")
  # print(f"Number images: {len(isic_paths)}")
  # print(f"Number malignant images: {np.sum(isic_dx == 1)}")
  # print(f"Number benign images: {np.sum(isic_dx == 0)}")

  assert len(isic_paths) == len(isic_ids) == len(isic_dx) == len(isic_skin_tone) == len(isic_patient_ids)

  # MILK
  print("Getting MILK...")
  df = pd.read_csv(f"{skin_tone_path}/Representative/Milk10K/metadata.csv")
  # Remove rows where dx is NaN or Indeterminate
  df = df.dropna(subset=['diagnosis_1'])
  df = df[df['diagnosis_1'] != 'Indeterminate']

  milk_ids = df["isic_id"].values
  milk_dx = df["diagnosis_1"].map({'Malignant': 1, 'Benign': 0}).values
  milk_skin_tone = df["skin_tone_class"].map({5: 0, 4: 0, 3: 1, 2: 1, 1: 2, 0: 2}).values
  milk_patient_ids = df["lesion_id"].values

  milk_paths = []
  for milk_id in milk_ids:
    milk_paths.append(f"{skin_tone_path}/Representative/Milk10K/images/{milk_id}.jpg")

  # print("----- MILK -----")
  # print(f"Number images: {len(milk_paths)}")
  # print(f"Number malignant images: {np.sum(milk_dx == 1)}")
  # print(f"Number benign images: {np.sum(milk_dx == 0)}")
  # print(f"Number skin tones: {np.unique(milk_skin_tone)}")
  # unique, counts = np.unique(milk_skin_tone, return_counts=True)
  # for i in range(len(unique)):
  #   print(f"Number images with skin tone {unique[i]}: {counts[i]}")

  assert len(milk_paths) == len(milk_ids) == len(milk_dx) == len(milk_skin_tone) == len(milk_patient_ids)

  print("Combining Datasets...")
  paths = np.concatenate((isic_paths, milk_paths))
  dx = np.concatenate((isic_dx, milk_dx))
  skin_tone = np.concatenate((isic_skin_tone, milk_skin_tone))
  patient_ids = np.concatenate((isic_patient_ids, milk_patient_ids))

  # print("---- Combined Datasets -----")
  # print(f"Number images: {len(paths)}")
  # print(f"Number malignant images: {np.sum(dx == 1)}")
  # print(f"Number benign images: {np.sum(dx == 0)}")
  strata = np.array([f"{d}_{s}" for d, s in zip(dx, skin_tone)])
  unique, counts = np.unique(strata, return_counts=True)
  for i in range(len(unique)):
    print(f"Number images with skin tone {unique[i]}: {counts[i]}")

  assert len(paths) == len(dx) == len(skin_tone)== len(patient_ids)

  return paths, dx, skin_tone, patient_ids

def get_train_isic2018():
  print("ISIC 2018 Training Dataset...")
  df = pd.read_csv(f"{isic2018_path}/metadata.csv")
  df = df.dropna(subset=['diagnosis_1'])

  isic_ids = df["isic_id"].values
  isic_dx = df["diagnosis_1"].map({'Malignant': 1, 'Benign': 0, 'Indeterminate': 1}).values
  isic_patient_ids = df["lesion_id"].values

  isic_paths = []
  for isic_id in isic_ids:
    isic_paths.append(f"{isic2018_path}/{isic_id}.jpg")

  assert len(isic_paths) == len(isic_dx) == len(isic_patient_ids)

  return isic_paths, isic_dx, isic_patient_ids

def get_test_isic2018():
  print("ISIC 2018 Test Dataset...")
  df = pd.read_csv(f"{isic2018_test_path}/metadata.csv")
  df = df.dropna(subset=['diagnosis_1'])

  isic_ids = df["isic_id"].values
  isic_dx = df["diagnosis_1"].map({'Malignant': 1, 'Benign': 0, 'Indeterminate': 1}).values

  isic_paths = []
  for isic_id in isic_ids:
    isic_paths.append(f"{isic2018_test_path}/{isic_id}.jpg")

  assert len(isic_paths) == len(isic_dx)

  return isic_paths, isic_dx

get_fitzpatrick_datasets()

Getting ISIC...
Getting MILK...
Combining Datasets...
Number images with skin tone 0_0: 5146
Number images with skin tone 0_1: 4201
Number images with skin tone 0_2: 1674
Number images with skin tone 1_0: 3732
Number images with skin tone 1_1: 5521
Number images with skin tone 1_2: 81


(array(['/root/.cache/kagglehub/datasets/josephgalicinao/skin-tone-dataset/versions/2/Representative/ISIC_archive/ISIC_0076337.jpg',
        '/root/.cache/kagglehub/datasets/josephgalicinao/skin-tone-dataset/versions/2/Representative/ISIC_archive/ISIC_0077599.jpg',
        '/root/.cache/kagglehub/datasets/josephgalicinao/skin-tone-dataset/versions/2/Representative/ISIC_archive/ISIC_0079358.jpg',
        ...,
        '/root/.cache/kagglehub/datasets/josephgalicinao/skin-tone-dataset/versions/2/Representative/Milk10K/images/ISIC_0075884.jpg',
        '/root/.cache/kagglehub/datasets/josephgalicinao/skin-tone-dataset/versions/2/Representative/Milk10K/images/ISIC_0073863.jpg',
        '/root/.cache/kagglehub/datasets/josephgalicinao/skin-tone-dataset/versions/2/Representative/Milk10K/images/ISIC_0051817.jpg'],
       dtype='<U123'),
 array([1, 0, 0, ..., 0, 0, 1]),
 array([0, 1, 0, ..., 1, 1, 0]),
 array(['IP_7309176', 'IP_5007819', 'IP_9328831', ..., 'IL_9270970',
        'IL_2547802', 'I

In [ ]:
def oversample_minority(x_train, y_train, abcde=None, weights=None):
    x_train = np.array(x_train)
    y_train = np.array(y_train)

    # Oversample the minority class
    pos_features = x_train[y_train == 1] # Get the malignant images
    neg_features = x_train[y_train == 0] # Get the benign images

    pos_labels = y_train[y_train == 1] # Get the malignant labels
    neg_labels = y_train[y_train == 0] # Get the benign labels

    if abcde is not None:
      pos_abcde = abcde[y_train == 1] # Get the malignant abcde features
      neg_abcde = abcde[y_train == 0] # Get the benign abcde features

    if weights is not None:
      pos_weights = weights[y_train == 1] # Get the malignant weights
      neg_weights = weights[y_train == 0] # Get the benign weights

    ids = np.arange(len(pos_features)) # Create an array that goes from 0 - len(pos_features)
    choice = np.random.choice(ids, len(neg_features)) # Choose len(neg_features) number of ids
    oversampled_pos_features = pos_features[choice] # Oversample the minority class to get images
    oversampled_pos_labels = pos_labels[choice] # Oversample the minority class to get labels
    if abcde is not None:
      oversampled_pos_abcde = pos_abcde[choice] # Oversample the minority class to get abcde features
    if weights is not None:
      oversampled_pos_weights = pos_weights[choice] # Oversample the minority class to get weights

    x_train = np.concatenate([oversampled_pos_features, neg_features], axis=0) # Concatenate the data sets
    y_train = np.concatenate([oversampled_pos_labels, neg_labels], axis=0) # Concatenate the data set

    if abcde is not None:
      abcde = np.concatenate([oversampled_pos_abcde, neg_abcde], axis=0) # Concatenate the
    if weights is not None:
      weights = np.concatenate([oversampled_pos_weights, neg_weights], axis=0) # Concatenate the data sets

    order = np.arange(len(x_train)) # Create an array that goes from 0 - len(x_train)
    np.random.shuffle(order) # Shuffle the array
    x_train = x_train[order] # Shuffle the images
    y_train = y_train[order] # Shuffle the labels

    if abcde is not None:
      abcde = abcde[order] # Shuffle the abcde features
    if weights is not None:
      weights = weights[order] # Shuffle the weights

    return x_train, y_train, abcde, weights

batch = 64

In [ ]:
def preprocess(image_processor):
  train_transform = T.Compose([
      T.Resize((224, 224)),
      T.RandomHorizontalFlip(p=0.5),
      T.RandomVerticalFlip(p=0.5),
      T.RandomApply([
          T.ColorJitter(
              brightness=0.1,
              contrast=0.1,
              saturation=0.1,
              hue=0.05
          )
      ], p=0.5),
      T.ToTensor(),
      T.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
  ])

  val_transform = T.Compose([
      T.Resize((224, 224)),
      T.ToTensor(),
      T.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
      T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0, inplace=False),
  ])

  return train_transform, val_transform

# Pretrained Models

### Init

In [ ]:
# Stratifier
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

from PIL import Image

# Dataset class
class SkinLesionDataset(Dataset):
    def __init__(self, labels, img_paths, transform=None):
        self.labels = labels
        self.img_paths = img_paths
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.open(self.img_paths[idx]).convert("RGB")
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        return {
            "pixel_values": image,
            "labels": label,
        }



### Metrics

In [ ]:
!pip install evaluate
import evaluate
import numpy as np
from sklearn.metrics import average_precision_score

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    probs = np.exp(logits) / np.exp(logits).sum(-1, keepdims=True)

    metrics = {}
    metrics.update(accuracy.compute(predictions=preds, references=labels))
    metrics.update(precision.compute(predictions=preds, references=labels, average="binary"))
    metrics.update(recall.compute(predictions=preds, references=labels, average="binary"))
    metrics.update(f1.compute(predictions=preds, references=labels, average="binary"))

    metrics["pr_auc"] = average_precision_score(labels, probs[:, 1])

    return metrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00


## ISIC 2018

Fine tune

In [ ]:
model_name = "google/vit-base-patch16-224"

img_paths, dx, patient_ids = get_train_isic2018()
img_paths = np.array(img_paths)
dx = np.array(dx)

with open('/content/drive/MyDrive/Thesis/isic2018_fine_tuning_vit.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  for prop in [0.75, 0.5, 0.25, 0]:
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(img_paths, dx, patient_ids)):
      train_imgs, val_imgs = img_paths[train_idx], img_paths[val_idx]
      train_dx, val_dx = dx[train_idx], dx[val_idx]
      train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)

      # Get Preprocessed Data
      image_processor = AutoImageProcessor.from_pretrained(model_name)
      train_transform, val_transform = preprocess(image_processor)
      train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
      val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

      print(f"Prop: {prop}")
      model = AutoModelForImageClassification.from_pretrained(
          model_name,
          num_labels=2,
          ignore_mismatched_sizes=True
      )

      # Freeze the whole ViT backbone first
      for param in model.vit.parameters():
          param.requires_grad = False

      layers = model.vit.encoder.layer
      n_layers = len(layers)
      print(f"Number of layers: {n_layers}")
      n_unfreeze = max(1, int(n_layers * prop))

      for layer in layers[-n_unfreeze:]:
          for param in layer.parameters():
              param.requires_grad = True

      # Keep classifier trainable
      for param in model.classifier.parameters():
          param.requires_grad = True

      # Train the model
      training_args = TrainingArguments(
          output_dir="./trains",
          per_device_train_batch_size=64,
          per_device_eval_batch_size=64,
          eval_strategy="epoch",
          save_strategy="epoch",
          logging_strategy="epoch",
          num_train_epochs=10,
          learning_rate=1e-5,
          save_total_limit=2,
          remove_unused_columns=False,
          load_best_model_at_end=True,
          metric_for_best_model="pr_auc",
          greater_is_better=True,
          dataloader_num_workers=8,
          dataloader_pin_memory=True,
          fp16=torch.cuda.is_available(),
          report_to="none",
          disable_tqdm=False,
      )

      trainer = Trainer(
          model=model,
          args=training_args,
          train_dataset=train_ds,
          eval_dataset=val_ds,
          compute_metrics=compute_metrics
      )

      trainer.train()

      eval_results = trainer.evaluate()

      print(eval_results)

      writer.writerow([
          prop,
          eval_results.get("eval_accuracy"),
          eval_results.get("eval_precision"),
          eval_results.get("eval_recall"),
          eval_results.get("eval_f1"),
          eval_results.get("eval_pr_auc"),
      ])

ISIC 2018 Training Dataset...


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Prop: 0.75


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.36656680703163147, 'eval_accuracy': 0.82901296111665, 'eval_precision': 0.5398671096345515, 'eval_recall': 0.8312020460358056, 'eval_f1': 0.6545820745216515, 'eval_pr_auc': 0.7090680969210164, 'eval_runtime': 3.9222, 'eval_samples_per_second': 511.45, 'eval_steps_per_second': 8.159, 'epoch': 10.0}


Evaluation Results

In [ ]:
train_imgs, train_dx = get_train_isic2018()
train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)
val_imgs, val_dx = get_test_isic2018()

train_imgs = np.array(train_imgs)
train_dx = np.array(train_dx)
val_imgs = np.array(val_imgs)
val_dx = np.array(val_dx)

prop = 0.3

with open('/content/drive/MyDrive/Thesis/isic2018_fine_tuning_vit.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
  val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

  print(f"Prop: {prop}")
  model = ViTForImageClassification.from_pretrained(
      model_name,
      num_labels=2,
      ignore_mismatched_sizes=True
  )

  # Freeze the whole ViT backbone first
  for param in model.vit.parameters():
      param.requires_grad = False

  layers = model.vit.encoder.layer
  n_layers = len(layers)
  print(f"Number of layers: {n_layers}")
  n_unfreeze = max(1, int(n_layers * prop))

  for layer in layers[-n_unfreeze:]:
      for param in layer.parameters():
          param.requires_grad = True

  # Keep classifier trainable
  for param in model.classifier.parameters():
      param.requires_grad = True

  # Train the model
  training_args = TrainingArguments(
      output_dir="./trains",
      per_device_train_batch_size=64,
      per_device_eval_batch_size=64,
      eval_strategy="epoch",
      save_strategy="epoch",
      logging_strategy="epoch",
      num_train_epochs=10,
      learning_rate=1e-5,
      save_total_limit=2,
      remove_unused_columns=False,
      load_best_model_at_end=True,
      metric_for_best_model="pr_auc",
      greater_is_better=True,
      dataloader_num_workers=8,
      dataloader_pin_memory=True,
      fp16=torch.cuda.is_available(),
      report_to="none",
      disable_tqdm=False,
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_ds,
      eval_dataset=val_ds,
      compute_metrics=compute_metrics
  )

  trainer.train()

  eval_results = trainer.evaluate()

  print(eval_results)

  writer.writerow([
      prop,
      eval_results.get("eval_accuracy"),
      eval_results.get("eval_precision"),
      eval_results.get("eval_recall"),
      eval_results.get("eval_f1"),
      eval_results.get("eval_pr_auc"),
  ])

ISIC 2018 Training Dataset...
ISIC 2018 Test Dataset...
Prop: 0.3


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.3698216676712036, 'eval_accuracy': 0.8379629629629629, 'eval_precision': 0.5756097560975609, 'eval_recall': 0.7687296416938111, 'eval_f1': 0.6582984658298466, 'eval_pr_auc': 0.7222507381218515, 'eval_runtime': 3.207, 'eval_samples_per_second': 471.465, 'eval_steps_per_second': 7.484, 'epoch': 10.0}


### Swin Transformers

Fine Tuning

In [ ]:
model_names = ["microsoft/swin-base-patch4-window7-224-in22k",
               "microsoft/swin-small-patch4-window7-224",
               "microsoft/swin-tiny-patch4-window7-224"]

img_paths, dx = get_train_isic2018()
img_paths = np.array(img_paths)
dx = np.array(dx)

with open('/content/drive/MyDrive/Thesis/isic2018_fine_tuning_swin.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  for model_name in model_names:
    for prop in [0.75, 0.5, 0.25, 0]:
      print(f"Prop: {prop}")
      for fold, (train_idx, val_idx) in enumerate(sgkf.split(img_paths, dx, img_paths)):
        train_imgs, val_imgs = img_paths[train_idx], img_paths[val_idx]
        train_dx, val_dx = dx[train_idx], dx[val_idx]
        train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)

        # Get Preprocessed Data
        image_processor = AutoImageProcessor.from_pretrained(model_name)
        train_transform, val_transform = preprocess(image_processor)
        train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
        val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

        model = AutoModelForImageClassification.from_pretrained(
            model_name,
            num_labels=2,
            ignore_mismatched_sizes=True
        )

        for param in model.swin.parameters():
            param.requires_grad = False

        # Flatten all blocks across stages
        all_blocks = []
        for stage in model.swin.encoder.layers:
            all_blocks.extend(stage.blocks)  # 'blocks' contains the transformer blocks

        n_blocks = len(all_blocks)
        print(f"Number of transformer blocks: {n_blocks}")

        n_unfreeze = max(1, int(n_blocks * prop))  # e.g., prop=0.25
        for block in all_blocks[-n_unfreeze:]:
            for param in block.parameters():
                param.requires_grad = True

        # Keep classifier trainable
        for param in model.classifier.parameters():
            param.requires_grad = True

        # Train the model
        training_args = TrainingArguments(
            output_dir="./trains",
            per_device_train_batch_size=64,
            per_device_eval_batch_size=64,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_strategy="epoch",
            num_train_epochs=10,
            learning_rate=1e-5,
            save_total_limit=2,
            remove_unused_columns=False,
            load_best_model_at_end=True,
            metric_for_best_model="pr_auc",
            greater_is_better=True,
            dataloader_num_workers=8,
            dataloader_pin_memory=True,
            fp16=torch.cuda.is_available(),
            report_to="none",
            disable_tqdm=False,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_metrics
        )

        trainer.train()

        eval_results = trainer.evaluate()

        print(eval_results)

        writer.writerow([model_name] + [
            prop,
            eval_results.get("eval_accuracy"),
            eval_results.get("eval_precision"),
            eval_results.get("eval_recall"),
            eval_results.get("eval_f1"),
            eval_results.get("eval_pr_auc"),
        ])

ISIC 2018 Training Dataset...
Prop: 0.3


Loading weights:   0%|          | 0/449 [00:00<?, ?it/s]

SwinForImageClassification LOAD REPORT from: microsoft/swin-base-patch4-window7-224-in22k
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([21841]) vs model:torch.Size([2])            
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([21841, 1024]) vs model:torch.Size([2, 1024])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Number of transformer blocks: 24


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Pr Auc
1,0.419173,0.377004,0.821767,0.531490,0.869347,0.659676,0.718830
2,0.311266,0.413212,0.813280,0.517241,0.904523,0.658135,0.766090
3,0.255094,0.328122,0.854718,0.591141,0.871859,0.704569,0.793046
4,0.207802,0.300273,0.872192,0.632463,0.851759,0.725910,0.807118
5,0.175853,0.291369,0.882676,0.658252,0.851759,0.742607,0.821008
6,0.149790,0.264731,0.895657,0.697286,0.839196,0.761688,0.825853
7,0.130802,0.264886,0.898153,0.708155,0.829146,0.763889,0.833482
8,0.115498,0.270783,0.897154,0.701681,0.839196,0.764302,0.836116
9,0.107363,0.273877,0.898153,0.707265,0.831658,0.764434,0.837578
10,0.102632,0.271484,0.897654,0.708423,0.824121,0.761905,0.837858


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.3838196098804474, 'eval_accuracy': 0.8052920619071393, 'eval_precision': 0.514866979655712, 'eval_recall': 0.80440097799511, 'eval_f1': 0.6278625954198473, 'eval_pr_auc': 0.7184951015074282, 'eval_runtime': 3.8156, 'eval_samples_per_second': 524.948, 'eval_steps_per_second': 8.387, 'epoch': 10.0}


Evaluate

In [ ]:
model_names = ["microsoft/swin-base-patch4-window7-224-in22k",
               "microsoft/swin-small-patch4-window7-224",
               "microsoft/swin-tiny-patch4-window7-224"]


train_imgs, train_dx = get_train_isic2018()
train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)
val_imgs, val_dx = get_test_isic2018()

train_imgs = np.array(train_imgs)
train_dx = np.array(train_dx)
val_imgs = np.array(val_imgs)
val_dx = np.array(val_dx)

prop = 0.3

with open('/content/drive/MyDrive/Thesis/isic2018_swin.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  for model_name in model_names:

    # Get Preprocessed Data
    image_processor = AutoImageProcessor.from_pretrained(model_name)
    train_transform, val_transform = preprocess(image_processor)
    train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
    val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

    model = AutoModelForImageClassification.from_pretrained(
        model_name,
        num_labels=2,
        ignore_mismatched_sizes=True
    )

    for param in model.swin.parameters():
        param.requires_grad = False

    # Flatten all blocks across stages
    all_blocks = []
    for stage in model.swin.encoder.layers:
        all_blocks.extend(stage.blocks)  # 'blocks' contains the transformer blocks

    n_blocks = len(all_blocks)
    print(f"Number of transformer blocks: {n_blocks}")

    n_unfreeze = max(1, int(n_blocks * prop))  # e.g., prop=0.25
    for block in all_blocks[-n_unfreeze:]:
        for param in block.parameters():
            param.requires_grad = True

    # Keep classifier trainable
    for param in model.classifier.parameters():
        param.requires_grad = True

    # Train the model
    training_args = TrainingArguments(
        output_dir="./trains",
        per_device_train_batch_size=64,
        per_device_eval_batch_size=64,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        num_train_epochs=10,
        learning_rate=1e-5,
        save_total_limit=2,
        remove_unused_columns=False,
        load_best_model_at_end=True,
        metric_for_best_model="pr_auc",
        greater_is_better=True,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        disable_tqdm=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()

    eval_results = trainer.evaluate()

    print(eval_results)

    writer.writerow([model_name] + [
        prop,
        eval_results.get("eval_accuracy"),
        eval_results.get("eval_precision"),
        eval_results.get("eval_recall"),
        eval_results.get("eval_f1"),
        eval_results.get("eval_pr_auc"),
    ])

Getting ISIC...
Getting MILK...
Combining Datasets...
Number images with skin tone 0_0: 5146
Number images with skin tone 0_1: 4201
Number images with skin tone 0_2: 1674
Number images with skin tone 1_0: 3732
Number images with skin tone 1_1: 5521
Number images with skin tone 1_2: 81


Loading weights:   0%|          | 0/449 [00:00<?, ?it/s]

SwinForImageClassification LOAD REPORT from: microsoft/swin-base-patch4-window7-224-in22k
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([21841, 1024]) vs model:torch.Size([2, 1024])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([21841]) vs model:torch.Size([2])            

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Number of transformer blocks: 24


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Pr Auc
1,0.326978,0.285492,0.882528,0.850026,0.896075,0.872443,0.926155


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

### DeiT

Fine Tuning

In [ ]:
model_names = ["facebook/deit-base-patch16-224",
                "facebook/deit-small-patch16-224",
                "facebook/deit-tiny-patch16-224"]

img_paths, dx = get_train_isic2018()
img_paths = np.array(img_paths)
dx = np.array(dx)

with open('/content/drive/MyDrive/Thesis/isic2018_fine_tuning_deit.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Model", "Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  for model_name in model_names:
    for prop in [0.75, 0.5, 0.25, 0]:
      for fold, (train_idx, val_idx) in enumerate(sgkf.split(img_paths, dx, img_paths)):
        train_imgs, val_imgs = img_paths[train_idx], img_paths[val_idx]
        train_dx, val_dx = dx[train_idx], dx[val_idx]
        train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)

        # Get Preprocessed Data
        image_processor = AutoImageProcessor.from_pretrained(model_name)
        train_transform, val_transform = preprocess(image_processor)
        train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
        val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

        print(f"Prop: {prop}")
        model = AutoModelForImageClassification.from_pretrained(
            model_name,
            num_labels=2,
            ignore_mismatched_sizes=True
        )

        # Freeze the whole ViT backbone first
        for param in model.vit.parameters():
            param.requires_grad = False

        layers = model.vit.encoder.layer
        n_layers = len(layers)
        print(f"Number of layers: {n_layers}")
        n_unfreeze = max(1, int(n_layers * prop))

        for layer in layers[-n_unfreeze:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Keep classifier trainable
        for param in model.classifier.parameters():
            param.requires_grad = True

        # Train the model
        training_args = TrainingArguments(
            output_dir="./trains",
            per_device_train_batch_size=64,
            per_device_eval_batch_size=64,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_strategy="epoch",
            num_train_epochs=10,
            learning_rate=1e-5,
            save_total_limit=2,
            remove_unused_columns=False,
            load_best_model_at_end=True,
            metric_for_best_model="pr_auc",
            greater_is_better=True,
            dataloader_num_workers=8,
            dataloader_pin_memory=True,
            fp16=torch.cuda.is_available(),
            report_to="none",
            disable_tqdm=False,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_metrics
        )

        trainer.train()

        eval_results = trainer.evaluate()

        print(eval_results)

        writer.writerow([
            model_name,
            prop,
            eval_results.get("eval_accuracy"),
            eval_results.get("eval_precision"),
            eval_results.get("eval_recall"),
            eval_results.get("eval_f1"),
            eval_results.get("eval_pr_auc"),
        ])

ISIC 2018 Training Dataset...


ValueError: too many values to unpack (expected 2)

Evaluate

In [ ]:
model_names = ["facebook/deit-base-patch16-224",
                "facebook/deit-small-patch16-224",
                "facebook/deit-tiny-patch16-224"]

train_imgs, train_dx = get_train_isic2018()
train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)
val_imgs, val_dx = get_test_isic2018()

train_imgs = np.array(train_imgs)
train_dx = np.array(train_dx)
val_imgs = np.array(val_imgs)
val_dx = np.array(val_dx)

prop = 0.3

with open('/content/drive/MyDrive/Thesis/isic2018_deit.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Model", "Precision", "Recall", "F1", "PR-AUC"])

  for model_name in model_names:
    # Get Preprocessed Data
    image_processor = AutoImageProcessor.from_pretrained(model_name)
    train_transform, val_transform = preprocess(image_processor)
    train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
    val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

    print(f"Prop: {prop}")
    model = AutoModelForImageClassification.from_pretrained(
        model_name,
        num_labels=2,
        ignore_mismatched_sizes=True
    )

    # Freeze the whole ViT backbone first
    for param in model.vit.parameters():
        param.requires_grad = False

    layers = model.vit.encoder.layer
    n_layers = len(layers)
    print(f"Number of layers: {n_layers}")
    n_unfreeze = max(1, int(n_layers * prop))

    for layer in layers[-n_unfreeze:]:
        for param in layer.parameters():
            param.requires_grad = True

    # Keep classifier trainable
    for param in model.classifier.parameters():
        param.requires_grad = True

    # Train the model
    training_args = TrainingArguments(
        output_dir="./trains",
        per_device_train_batch_size=64,
        per_device_eval_batch_size=64,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        num_train_epochs=10,
        learning_rate=1e-5,
        save_total_limit=2,
        remove_unused_columns=False,
        load_best_model_at_end=True,
        metric_for_best_model="pr_auc",
        greater_is_better=True,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        disable_tqdm=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()

    eval_results = trainer.evaluate()

    print(eval_results)

    writer.writerow([
        model_name,
        eval_results.get("eval_precision"),
        eval_results.get("eval_recall"),
        eval_results.get("eval_f1"),
        eval_results.get("eval_pr_auc"),
    ])

ISIC 2018 Training Dataset...
ISIC 2018 Test Dataset...


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Prop: 0.3


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.4467390775680542, 'eval_accuracy': 0.7896825396825397, 'eval_precision': 0.4892367906066536, 'eval_recall': 0.8143322475570033, 'eval_f1': 0.6112469437652812, 'eval_pr_auc': 0.622017376591974, 'eval_runtime': 1.899, 'eval_samples_per_second': 796.205, 'eval_steps_per_second': 12.638, 'epoch': 10.0}


# Adversarial Network

## Gradient Reversal Layer

In [ ]:
from torch.autograd import Function
import torch

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

class GradReverse(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None

def grad_reverse(x, lambda_=1.0):
    return GradReverse.apply(x, lambda_)

## Metrics

In [ ]:
!pip install evaluate
import evaluate
import numpy as np
from sklearn.metrics import average_precision_score

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, label_ids = eval_pred

    main_logits, adv_logits = predictions
    main_labels, skin_labels = label_ids

    main_preds = np.argmax(main_logits, axis=1)
    adv_preds = np.argmax(adv_logits, axis=1)

    print("Adv pred counts:", np.bincount(adv_preds))
    print("Adv true counts:", np.bincount(skin_labels))

    # stable softmax for PR-AUC
    shifted = main_logits - np.max(main_logits, axis=1, keepdims=True)
    main_probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)

    metrics = {}

    # Main task metrics
    metrics["accuracy"] = accuracy.compute(
        predictions=main_preds, references=main_labels
    )["accuracy"]
    metrics["precision"] = precision.compute(
        predictions=main_preds, references=main_labels, average="binary"
    )["precision"]
    metrics["recall"] = recall.compute(
        predictions=main_preds, references=main_labels, average="binary"
    )["recall"]
    metrics["f1"] = f1.compute(
        predictions=main_preds, references=main_labels, average="binary"
    )["f1"]
    metrics["pr_auc"] = average_precision_score(main_labels, main_probs[:, 1])

    # Adversarial task metrics
    metrics["adv_accuracy"] = accuracy.compute(
        predictions=adv_preds, references=skin_labels
    )["accuracy"]

    return metrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00


## Prepare Data

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
class AdversarialSkinLesionDataset(Dataset):
    def __init__(self, labels, skin_tones, img_paths, transform=None):
        self.labels = labels
        self.img_paths = img_paths
        self.transform = transform
        self.skin_tones = skin_tones

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.open(self.img_paths[idx]).convert("RGB")
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        skin_tone = torch.tensor(self.skin_tones[idx], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        return {
            "pixel_values": image,
            "labels": label,
            "skin_tone_labels": skin_tone
        }

from transformers import Trainer

class AdversarialTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(
            pixel_values=inputs["pixel_values"],
            labels=inputs.get("labels"),
            skin_tone_labels=inputs.get("skin_tone_labels"),
        )
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        has_labels = "labels" in inputs and "skin_tone_labels" in inputs

        inputs = self._prepare_inputs(inputs)

        with torch.no_grad():
            outputs = model(
                pixel_values=inputs["pixel_values"],
                labels=inputs.get("labels"),
                skin_tone_labels=inputs.get("skin_tone_labels"),
            )

        loss = outputs["loss"].detach() if has_labels and outputs.get("loss") is not None else None

        if prediction_loss_only:
            return (loss, None, None)

        logits = (
            outputs["logits"].detach(),
            outputs["adversarial_logits"].detach(),
        )

        labels = None
        if has_labels:
            labels = (
                inputs["labels"].detach(),
                inputs["skin_tone_labels"].detach(),
            )

        return (loss, logits, labels)

## Swin Model with Adversarial Network

In [ ]:
import torch
from transformers import AutoModelForImageClassification

class SwinModelWithAdversarial(torch.nn.Module):
    def __init__(
        self,
        model_name,
        prop=0.75,
        num_labels=2,
        adv_classes=5,
        lambda_=0.0,
        loss_fn=torch.nn.CrossEntropyLoss(),
    ):
        super().__init__()

        self.model = AutoModelForImageClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            output_hidden_states=True,
            ignore_mismatched_sizes=True,
            output_attentions=True
        )

        self.loss_fn = loss_fn
        self.lambda_ = lambda_

        for param in self.model.swin.parameters():
            param.requires_grad = False

        all_blocks = []
        for stage in self.model.swin.encoder.layers:
            all_blocks.extend(stage.blocks)

        n_blocks = len(all_blocks)
        n_unfreeze = max(1, int(n_blocks * prop))

        for block in all_blocks[-n_unfreeze:]:
            for param in block.parameters():
                param.requires_grad = True

        for param in self.model.classifier.parameters():
            param.requires_grad = True

        hidden_size = self.model.config.hidden_size

        self.adversarial = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, 128),
            torch.nn.GELU(),
            torch.nn.Linear(128, adv_classes),
        )

    def forward(
        self,
        pixel_values,
        labels=None,
        skin_tone_labels=None,
        output_attentions=False,
        output_hidden_states=True,
        return_dict=True,
    ):
        outputs = self.model(
            pixel_values=pixel_values,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        features = outputs.hidden_states[-1].mean(dim=1)
        logits = outputs.logits

        reverse_features = grad_reverse(features, lambda_=self.lambda_)
        adversarial_logits = self.adversarial(reverse_features)

        prediction_loss = None
        adversarial_loss = None
        total_loss = None

        if labels is not None:
            labels = labels.view(-1).long()
            prediction_loss = self.loss_fn(logits, labels)

        if skin_tone_labels is not None:
            skin_tone_labels = skin_tone_labels.view(-1).long()
            adversarial_loss = self.loss_fn(adversarial_logits, skin_tone_labels)

        if prediction_loss is not None and adversarial_loss is not None:
            total_loss = prediction_loss + adversarial_loss
        elif prediction_loss is not None:
            total_loss = prediction_loss

        return {
            "loss": total_loss,
            "logits": logits,
            "adversarial_logits": adversarial_logits,
            "attentions": outputs.attentions,
            "hidden_states": outputs.hidden_states,
        }

Fine Tune Lambda

In [ ]:

from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference, equal_opportunity_difference
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

NUM_CLASSES = 3

model_name = "microsoft/swin-tiny-patch4-window7-224"

img_paths, dx, skin_tones, patient_ids = get_fitzpatrick_datasets()

# Convert to numpy arrays so indexing by split indices is easy
img_paths = np.array(img_paths)
dx = np.array(dx)
skin_tones = np.array(skin_tones)
patient_ids = np.array(patient_ids)

# Stratify on diagnosis + skin tone
strata = np.array([f"{d}_{s}" for d, s in zip(dx, skin_tones)])

# -------------------------
# 1) Split off TEST set
# -------------------------
# 5 folds -> one fold is 20% test
sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
trainval_idx, test_idx = next(sgkf_test.split(img_paths, strata, groups=patient_ids))

trainval_imgs = img_paths[trainval_idx]
trainval_dx = dx[trainval_idx]
trainval_skin_tones = skin_tones[trainval_idx]
trainval_patient_ids = patient_ids[trainval_idx]

trainval_strata = np.array([f"{d}_{s}" for d, s in zip(trainval_dx, trainval_skin_tones)])

# 4 folds -> one fold is 25% of trainval = 20% of original
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

with open(f'/content/drive/MyDrive/Thesis/swin_adv_fine_tune_{NUM_CLASSES}.csv', 'a', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Lambda", "Precision", "Recall", "F1", "PR-AUC", "Adversarial Accuracy", "Equal Opportuinity",
                   "Equalized Odds", "Demographic Parity"])

  # Get Preprocessed Data
  for lambda_ in [0.03, 0.04, 0.05, 0, 0.01, 0.02]:
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(trainval_imgs, trainval_strata, trainval_patient_ids)):
        train_imgs, val_imgs = trainval_imgs[train_idx], trainval_imgs[val_idx]
        train_dx, val_dx = trainval_dx[train_idx], trainval_dx[val_idx]
        train_skin_tones, val_skin_tones = trainval_skin_tones[train_idx], trainval_skin_tones[val_idx]

        image_processor = AutoImageProcessor.from_pretrained(model_name)
        train_transform, val_transform = preprocess(image_processor)
        train_ds = AdversarialSkinLesionDataset(train_dx, train_skin_tones, train_imgs, transform=train_transform)
        val_ds = AdversarialSkinLesionDataset(val_dx, val_skin_tones, val_imgs, transform=val_transform)

        model = SwinModelWithAdversarial(model_name, lambda_=lambda_, adv_classes=NUM_CLASSES)

        # Train the model
        training_args = TrainingArguments(
            output_dir="./trains",
            per_device_train_batch_size=64,
            per_device_eval_batch_size=64,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_strategy="epoch",
            num_train_epochs=5,
            learning_rate=1e-5,
            save_total_limit=2,
            remove_unused_columns=False,
            load_best_model_at_end=True,
            metric_for_best_model="pr_auc",
            greater_is_better=True,
            dataloader_num_workers=8,
            dataloader_pin_memory=True,
            fp16=torch.cuda.is_available(),
            report_to="none",
            disable_tqdm=False,
        )

        trainer = AdversarialTrainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_metrics
        )

        # Get the results on the validation dataset
        trainer.train()
        eval_results = trainer.evaluate()

        # Get the results for the light, medium, and dark datasets
        val_results = trainer.predict(val_ds)

        val_pred = np.argmax(val_results.predictions[0], axis=1)
        val_dx = np.array(val_dx)

        equal_opportunity = equal_opportunity_difference(val_dx, val_pred, sensitive_features=val_skin_tones)
        demographic_parity = demographic_parity_difference(val_dx, val_pred, sensitive_features=val_skin_tones)
        equalized_odds = equalized_odds_difference(val_dx, val_pred, sensitive_features=val_skin_tones, agg="mean")

        # Failsafe in case the runtime breaks
        print(eval_results)
        print("Equal Opportunity:", equal_opportunity)
        print("Demographic Parity:", demographic_parity)
        print("Equalized Odds:", equalized_odds)

        writer.writerow([
          model_name,
          lambda_,
          eval_results.get("eval_precision"),
          eval_results.get("eval_recall"),
          eval_results.get("eval_f1"),
          eval_results.get("eval_pr_auc"),
          eval_results.get("eval_adv_accuracy"),
          equal_opportunity,
          equalized_odds,
          demographic_parity,
        ])

Getting ISIC...
Getting MILK...
Combining Datasets...
Number images with skin tone 0_0: 5146
Number images with skin tone 0_1: 4201
Number images with skin tone 0_2: 1674
Number images with skin tone 1_0: 3732
Number images with skin tone 1_1: 5521
Number images with skin tone 1_2: 81


AttributeError: module 'torch.nn' has no attribute 'HingeLoss'

Evaluate

In [ ]:
from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference, equal_opportunity_difference
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
NUM_CLASSES = 2

model_name = "microsoft/swin-tiny-patch4-window7-224"

img_paths, dx, skin_tones, patient_ids = get_fitzpatrick_datasets()

# Convert to numpy arrays so indexing by split indices is easy
img_paths = np.array(img_paths)
dx = np.array(dx)
skin_tones = np.array(skin_tones)
patient_ids = np.array(patient_ids)

# Stratify on diagnosis + skin tone
strata = np.array([f"{d}_{s}" for d, s in zip(dx, skin_tones)])

# -------------------------
# 1) Split off TEST set
# -------------------------
# 5 folds -> one fold is 20% test
sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf_test.split(img_paths, strata, groups=patient_ids))

train_imgs = img_paths[train_idx]
val_imgs = img_paths[val_idx]

train_dx = dx[train_idx]
val_dx = dx[val_idx]

train_skin_tones = skin_tones[train_idx]
val_skin_tones = skin_tones[val_idx]

train_patient_ids = patient_ids[train_idx]
val_patient_ids = patient_ids[val_idx]

# -------------------------
# 3) Verify no patient leakage
# -------------------------
assert set(train_patient_ids).isdisjoint(set(val_patient_ids))

print("Train:", len(train_imgs))
print("Val:", len(val_imgs))
print("No patient overlap across splits.")

with open('/content/drive/MyDrive/Thesis/swin_adversarial_fine_tune.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Lambda", "Fold", "Precision", "Recall", "F1", "PR-AUC",
                   "Light Precision", "Light Recall", "Light F1", "Light PR-AUC",
                   "Medium Precision", "Medium Recall", "Medium F1", "Medium PR-AUC",
                   "Dark Precision", "Dark Recall", "Dark F1", "Dark PR-AUC",])

  # Get Preprocessed Data
  image_processor = AutoImageProcessor.from_pretrained(model_name)
  train_transform, val_transform = preprocess(image_processor)
  train_ds = AdversarialSkinLesionDataset(train_dx, train_skin_tones, train_imgs, transform=train_transform)
  val_ds = AdversarialSkinLesionDataset(val_dx, val_skin_tones, val_imgs, transform=val_transform)

  model = SwinModelWithAdversarial(model_name, lambda_=0.035, adv_classes=NUM_CLASSES)

  # Train the model
  training_args = TrainingArguments(
      output_dir="./trains",
      per_device_train_batch_size=64,
      per_device_eval_batch_size=64,
      eval_strategy="epoch",
      save_strategy="epoch",
      logging_strategy="epoch",
      num_train_epochs=10,
      learning_rate=1e-5,
      save_total_limit=2,
      remove_unused_columns=False,
      load_best_model_at_end=True,
      metric_for_best_model="pr_auc",
      greater_is_better=True,
      dataloader_num_workers=8,
      dataloader_pin_memory=True,
      fp16=torch.cuda.is_available(),
      report_to="none",
      disable_tqdm=False,
  )

  trainer = AdversarialTrainer(
      model=model,
      args=training_args,
      train_dataset=train_ds,
      eval_dataset=val_ds,
      compute_metrics=compute_metrics
  )

  # Get the results on the validation dataset
  trainer.train()
  eval_results = trainer.evaluate()

  # Get the results for the light, medium, and dark datasets
  val_results = trainer.predict(val_ds)
  val_pred = np.argmax(val_results.predictions[0], axis=1)
  val_dx = np.array(val_dx)

  equal_opportunity = equal_opportunity_difference(val_dx, val_pred, sensitive_features=val_skin_tones)
  demographic_parity = demographic_parity_difference(val_dx, val_pred, sensitive_features=val_skin_tones)
  equalized_odds = equalized_odds_difference(val_dx, val_pred, sensitive_features=val_skin_tones, agg="mean")

  print("Equal Opportunity:", equal_opportunity)
  print("Demographic Parity:", demographic_parity)
  print("Equalized Odds:", equalized_odds)

  # Failsafe in case the runtime breaks
  print(eval_results)

  writer.writerow([
    model_name,
    eval_results.get("eval_precision"),
    eval_results.get("eval_recall"),
    eval_results.get("eval_f1"),
    eval_results.get("eval_pr_auc"),
    eval_results.get("eval_adv_accuracy"),
    equal_opportunity,
    equalized_odds,
    demographic_parity,
  ])

  # Save the model
  trainer.save_model('/content/drive/MyDrive/Thesis/models/swin_adversarial_0.03')
  trainer.save_state()

Getting ISIC...
Getting MILK...
Combining Datasets...
Number images with skin tone 0_0: 7912
Number images with skin tone 0_1: 3109
Number images with skin tone 1_0: 8801
Number images with skin tone 1_1: 533
Train: 16746
Val: 3609
No patient overlap across splits.


Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Pr Auc,Adv Accuracy
1,0.822287,0.660867,0.867830,0.887701,0.861443,0.874375,0.921958,0.862843
2,0.661287,0.633026,0.872818,0.847209,0.929424,0.886414,0.932315,0.865891
3,0.626199,0.615916,0.883347,0.871302,0.916969,0.893552,0.935053,0.876420
4,0.605788,0.610839,0.886672,0.882947,0.908147,0.895370,0.935218,0.879468
5,0.599667,0.636566,0.884178,0.882413,0.903477,0.892821,0.937334,0.863951
6,0.623106,0.703795,0.879191,0.894236,0.877530,0.885804,0.935836,0.849820
7,0.690301,0.844000,0.881408,0.883376,0.896212,0.889748,0.938598,0.794957
8,0.819105,1.015417,0.883624,0.875436,0.911780,0.893238,0.939410,0.732336
9,0.984096,1.206358,0.881408,0.884951,0.894136,0.889520,0.939013,0.687171
10,1.101580,1.285042,0.881131,0.886880,0.891022,0.888946,0.938743,0.673871


Adv pred counts: [3557   52]
Adv true counts: [3084  525]
Adv pred counts: [3502  107]
Adv true counts: [3084  525]
Adv pred counts: [3454  155]
Adv true counts: [3084  525]
Adv pred counts: [3461  148]
Adv true counts: [3084  525]
Adv pred counts: [3503  106]
Adv true counts: [3084  525]
Adv pred counts: [3484  125]
Adv true counts: [3084  525]
Adv pred counts: [3302  307]
Adv true counts: [3084  525]
Adv pred counts: [3062  547]
Adv true counts: [3084  525]
Adv pred counts: [2895  714]
Adv true counts: [3084  525]
Adv pred counts: [2843  766]
Adv true counts: [3084  525]


Adv pred counts: [3062  547]
Adv true counts: [3084  525]
Adv pred counts: [3062  547]
Adv true counts: [3084  525]
Equal Opportunity: 0.20887564685588833
Demographic Parity: 0.4278765981100612
Equalized Odds: 0.16382013419694275
{'eval_loss': 1.0154168605804443, 'eval_accuracy': 0.8836242726517041, 'eval_precision': 0.8754359740906826, 'eval_recall': 0.9117799688635184, 'eval_f1': 0.8932384341637011, 'eval_pr_auc': 0.9394096868875713, 'eval_adv_accuracy': 0.7323358270989194, 'eval_runtime': 15.9459, 'eval_samples_per_second': 226.328, 'eval_steps_per_second': 3.575, 'epoch': 10.0}


Fariness Metrics

In [ ]:
img_paths, dx, skin_tones = get_fitzpatrick_datasets()
strata = [f"{a}_{b}" for a, b in zip(dx, skin_tones)]
train_imgs, val_imgs, train_dx, val_dx, train_skin_tones, val_skin_tones = train_test_split(img_paths, dx, skin_tones, test_size=0.2, random_state=42, stratify=strata)

model_names = [
    "swin_adversarial_0",
]

with open('/content/drive/MyDrive/Thesis/swin_fairness.csv', 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)

    for model_name in model_names:
        save_path = f"/content/drive/MyDrive/Thesis/models/{model_name}"
        lambda_value = float(model_name.split("_")[-1])

        image_processor = AutoImageProcessor.from_pretrained(save_path)
        train_transform, val_transform = preprocess(image_processor)
        inputs = val_transform(images=val_imgs, return_tensors="pt")

        # Recreate model
        model = SwinModelWithAdversarial("microsoft/swin-tiny-patch4-window7-224", lambda_=lambda_value)

        # Load weights
        if os.path.exists(f"{save_path}/model.safetensors"):
            from safetensors.torch import load_file
            state_dict = load_file(f"{save_path}/model.safetensors")
        elif os.path.exists(f"{save_path}/pytorch_model.bin"):
            state_dict = torch.load(f"{save_path}/pytorch_model.bin", map_location="cpu")
        else:
            raise FileNotFoundError(f"No model file found in {save_path}")

        model.load_state_dict(state_dict)
        outputs = model(**inputs)

        print(outputs.attentions)

Getting ISIC...
----- ISIC -----
Number images: 10121
Number malignant images: 2066
Number benign images: 8055
Number images with skin tone 0: 6012
Number images with skin tone 1: 2538
Number images with skin tone 2: 1571
Getting MILK...
Number images with skin tone 0: 2866
Number images with skin tone 1: 7184
Number images with skin tone 2: 184
----- MILK -----
Number images: 10234
Number malignant images: 7268
Number benign images: 2966
Number skin tones: [0 1 2]
Getting MILK
---- Combined Datasets -----
Number images: 20355
Number malignant images: 9334
Number benign images: 11021
Number images with skin tone 0: 8878
Number images with skin tone 1: 9722
Number images with skin tone 2: 1755


Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Adv pred counts: [1081  686    9]
Adv true counts: [1776]


Adv pred counts: [ 369 1517   58]
Adv true counts: [   0 1944]


Adv pred counts: [ 44  53 254]
Adv true counts: [  0   0 351]


# Oversampling

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
!pip install fairlearn
from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference, equal_opportunity_difference

model_name = "microsoft/swin-tiny-patch4-window7-224"

img_paths, dx, skin_tones, patient_ids = get_fitzpatrick_datasets()

# Convert to numpy arrays so indexing by split indices is easy
img_paths = np.array(img_paths)
dx = np.array(dx)
skin_tones = np.array(skin_tones)
patient_ids = np.array(patient_ids)

# Stratify on diagnosis + skin tone
strata = np.array([f"{d}_{s}" for d, s in zip(dx, skin_tones)])

# Create test dataset
sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf_test.split(img_paths, strata, groups=patient_ids))

train_imgs = img_paths[train_idx]
val_imgs = img_paths[val_idx]

train_dx = dx[train_idx]
val_dx = dx[val_idx]

train_skin_tones = skin_tones[train_idx]
val_skin_tones = skin_tones[val_idx]

train_patient_ids = patient_ids[train_idx]
val_patient_ids = patient_ids[val_idx]

# Ensures no patient id leakage
assert set(train_patient_ids).isdisjoint(set(val_patient_ids))

# Oversample such that the same number of images in each strata
# NOTE: Most benign classes is in the light dataset and the most malignant classes is in the medium dataset

# Make each dataset have the same number of benign cases
light_benign_idx = np.where((train_skin_tones == 0) & (train_dx == 0))[0]
medium_benign_idx = np.where((train_skin_tones == 1) & (train_dx == 0))[0]
dark_benign_idx = np.where((train_skin_tones == 2) & (train_dx == 0))[0]

light_medium_paths = np.concatenate([train_imgs[light_benign_idx], train_imgs[medium_benign_idx]])
light_medium_dx = np.concatenate([train_dx[light_benign_idx], train_dx[medium_benign_idx] + 1]) # Add one so that we oversample medium_dx, will change back to 0
light_medium_paths, light_medium_dx, _, _ = oversample_minority(light_medium_paths, light_medium_dx)
medium_benign_paths = light_medium_paths[light_medium_dx == 1]

light_dark_paths = np.concatenate([train_imgs[light_benign_idx], train_imgs[dark_benign_idx]])
light_dark_dx = np.concatenate([train_dx[light_benign_idx], train_dx[dark_benign_idx] + 1])
light_dark_paths, light_dark_dx, _, _ = oversample_minority(light_dark_paths, light_dark_dx)
dark_benign_paths = light_dark_paths[light_dark_dx == 1]

light_benign_paths = train_imgs[light_benign_idx]
print(f"Oversampled light benign: {len(light_benign_paths)}")
print(f"Oversampled medium benign: {len(medium_benign_paths)}")
print(f"Oversampled dark benign: {len(dark_benign_paths)}")

# Make each dataset have the same number of malignant classes
light_malignant_idx = np.where((train_skin_tones == 0) & (train_dx == 1))[0]
medium_malignant_idx = np.where((train_skin_tones == 1) & (train_dx == 1))[0]
dark_malignant_idx = np.where((train_skin_tones == 2) & (train_dx == 1))[0]

medium_light_paths = np.concatenate([train_imgs[medium_malignant_idx], train_imgs[light_malignant_idx]])
medium_light_dx = np.concatenate([train_dx[medium_malignant_idx] - 1, train_dx[light_malignant_idx]])
medium_light_paths, medium_light_dx, _, _ = oversample_minority(medium_light_paths, medium_light_dx)
light_malignant_paths = medium_light_paths[medium_light_dx == 0]

medium_dark_paths = np.concatenate([train_imgs[medium_malignant_idx], train_imgs[dark_malignant_idx]])
medium_dark_dx = np.concatenate([train_dx[medium_malignant_idx] - 1, train_dx[dark_malignant_idx]])
medium_dark_paths, medium_dark_dx, _, _ = oversample_minority(medium_dark_paths, medium_dark_dx)
dark_malignant_paths = medium_dark_paths[medium_dark_dx == 0]

medium_malignant_paths = train_imgs[medium_malignant_idx]
print(f"Oversampled medium malignant: {len(light_malignant_paths)}")
print(f"Oversampled medium malignant: {len(medium_malignant_paths)}")
print(f"Oversampled dark malignant: {len(dark_malignant_paths)}")

# Combine the datasets
benign_paths = np.concatenate([light_benign_paths, medium_benign_paths, dark_benign_paths])
benign_dx = np.zeros(len(benign_paths))
malignant_paths = np.concatenate([light_malignant_paths, medium_malignant_paths, dark_malignant_paths])
malignant_dx = np.ones(len(malignant_paths))
benign_skin_tones = np.concatenate([np.zeros(len(light_benign_paths)), np.ones(len(medium_benign_paths)), np.ones(len(dark_benign_paths)) * 2])
malignant_skin_tones = np.concatenate([np.zeros(len(light_malignant_paths)), np.ones(len(medium_malignant_paths)), np.ones(len(dark_malignant_paths)) * 2])

train_imgs = np.concatenate([benign_paths, malignant_paths])
train_dx = np.concatenate([benign_dx, malignant_dx])
train_skin_tones = np.concatenate([benign_skin_tones, malignant_skin_tones])

assert(len(train_imgs) == len(train_dx) == len(train_skin_tones))

with open('/content/drive/MyDrive/Thesis/swin_oversample_adversarial.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Lambda", "Fold", "Precision", "Recall", "F1", "PR-AUC",
                   "Light Precision", "Light Recall", "Light F1", "Light PR-AUC",
                   "Medium Precision", "Medium Recall", "Medium F1", "Medium PR-AUC",
                   "Dark Precision", "Dark Recall", "Dark F1", "Dark PR-AUC",])

  image_processor = AutoImageProcessor.from_pretrained(model_name)
  train_transform, val_transform = preprocess(image_processor)
  train_ds = AdversarialSkinLesionDataset(train_dx, train_skin_tones, train_imgs, transform=train_transform)
  val_ds = AdversarialSkinLesionDataset(val_dx, val_skin_tones, val_imgs, transform=val_transform)

  model = SwinModelWithAdversarial(model_name, lambda_=0.03)

  # Train the model
  training_args = TrainingArguments(
      output_dir="./trains",
      per_device_train_batch_size=64,
      per_device_eval_batch_size=64,
      eval_strategy="epoch",
      save_strategy="epoch",
      logging_strategy="epoch",
      num_train_epochs=10,
      learning_rate=1e-5,
      save_total_limit=2,
      remove_unused_columns=False,
      load_best_model_at_end=True,
      metric_for_best_model="pr_auc",
      greater_is_better=True,
      dataloader_num_workers=8,
      dataloader_pin_memory=True,
      fp16=torch.cuda.is_available(),
      report_to="none",
      disable_tqdm=False,
  )

  trainer = AdversarialTrainer(
      model=model,
      args=training_args,
      train_dataset=train_ds,
      eval_dataset=val_ds,
      compute_metrics=compute_metrics
  )

  # Get the results on the validation dataset
  trainer.train()
  eval_results = trainer.evaluate()

  val_results = trainer.predict(val_ds)
  val_pred = np.argmax(val_results.predictions[0], axis=1)
  val_dx = np.array(val_dx)

  equal_opportunity = equal_opportunity_difference(val_dx, val_pred, sensitive_features=val_skin_tones)
  demographic_parity = demographic_parity_difference(val_dx, val_pred, sensitive_features=val_skin_tones)
  equalized_odds = equalized_odds_difference(val_dx, val_pred, sensitive_features=val_skin_tones)

  # Failsafe in case the runtime breaks
  print(eval_results)

  writer.writerow([
    model_name,
    eval_results.get("eval_precision"),
    eval_results.get("eval_recall"),
    eval_results.get("eval_f1"),
    eval_results.get("eval_pr_auc"),
    equal_opportunity,
    equal_opportunity,
    demographic_parity,
    equalized_odds
  ])

Getting ISIC...
Getting MILK...
Number images with skin tone 0: 2866
Number images with skin tone 1: 7184
Number images with skin tone 2: 184
Combining Datasets...
Oversampled light benign: 4151
Oversampled medium benign: 4151
Oversampled dark benign: 4151
Oversampled medium malignant: 4447
Oversampled medium malignant: 4447
Oversampled dark malignant: 4447


The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Pr Auc,Adv Accuracy
1,1.372025,1.391112,0.881041,0.848453,0.894417,0.870829,0.907697,0.417348
2,1.316865,1.492364,0.881537,0.844283,0.902156,0.872261,0.910381,0.330359
3,1.685963,2.328829,0.880050,0.853333,0.884467,0.868621,0.907837,0.269145
4,2.758077,3.800320,0.882528,0.848564,0.898286,0.872718,0.912894,0.194548
5,3.551906,4.335867,0.877323,0.855905,0.873411,0.864569,0.910610,0.341264
6,2.472404,2.846300,0.879802,0.846960,0.893311,0.869518,0.910832,0.354399
7,1.804193,2.968343,0.880297,0.854925,0.882808,0.868643,0.910831,0.302107
8,2.784812,5.782259,0.880545,0.855003,0.883361,0.868951,0.911063,0.207187
9,4.737006,8.464162,0.878315,0.860503,0.869541,0.864999,0.913474,0.125898
10,6.037749,9.611159,0.878067,0.855370,0.876175,0.865647,0.913476,0.088724


Adv pred counts: [ 964 1499 1572]
Adv true counts: [1713 2061  261]
Adv pred counts: [1069 1469 1497]
Adv true counts: [1713 2061  261]
Adv pred counts: [ 812 1682 1541]
Adv true counts: [1713 2061  261]
Adv pred counts: [2257  171 1607]
Adv true counts: [1713 2061  261]
Adv pred counts: [ 521 2006 1508]
Adv true counts: [1713 2061  261]
Adv pred counts: [ 466 2536 1033]
Adv true counts: [1713 2061  261]
Adv pred counts: [ 499 1848 1688]
Adv true counts: [1713 2061  261]
Adv pred counts: [ 282 1120 2633]
Adv true counts: [1713 2061  261]
Adv pred counts: [ 245  628 3162]
Adv true counts: [1713 2061  261]
Adv pred counts: [ 231  411 3393]
Adv true counts: [1713 2061  261]


Adv pred counts: [ 231  411 3393]
Adv true counts: [1713 2061  261]
Adv pred counts: [ 231  411 3393]
Adv true counts: [1713 2061  261]
{'eval_loss': 9.611159324645996, 'eval_accuracy': 0.8780669144981412, 'eval_precision': 0.8553696708041014, 'eval_recall': 0.8761746821448314, 'eval_f1': 0.8656471873293282, 'eval_pr_auc': 0.9134755974799472, 'eval_adv_accuracy': 0.08872366790582403, 'eval_runtime': 21.2951, 'eval_samples_per_second': 189.48, 'eval_steps_per_second': 3.005, 'epoch': 10.0}


# Quantization

Load Swin Weights without Adversarial Model

In [ ]:
import os
import torch
from transformers import AutoImageProcessor, SwinForImageClassification

path = "/content/drive/MyDrive/Thesis/models/swin_adversarial_0.03"
base_model_name = "microsoft/swin-tiny-patch4-window7-224"

model = SwinForImageClassification.from_pretrained(
    base_model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
)
processor = AutoImageProcessor.from_pretrained(path)

state_path_bin = os.path.join(path, "pytorch_model.bin")
state_path_safe = os.path.join(path, "model.safetensors")

if os.path.exists(state_path_bin):
    state_dict = torch.load(state_path_bin, map_location="cpu")
elif os.path.exists(state_path_safe):
    from safetensors.torch import load_file
    state_dict = load_file(state_path_safe)
else:
    raise FileNotFoundError("No pytorch_model.bin or model.safetensors found")

if "state_dict" in state_dict and isinstance(state_dict["state_dict"], dict):
    state_dict = state_dict["state_dict"]
elif "model_state_dict" in state_dict and isinstance(state_dict["model_state_dict"], dict):
    state_dict = state_dict["model_state_dict"]

mapped_state = {}

for k, v in state_dict.items():
    if k.startswith("model.swin."):
        new_k = k.replace("model.swin.", "swin.", 1)
        mapped_state[new_k] = v
    elif k.startswith("model.classifier."):
        new_k = k.replace("model.", "", 1)   # -> classifier.weight / classifier.bias
        mapped_state[new_k] = v

missing, unexpected = model.load_state_dict(mapped_state, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

# model.eval()

Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-tiny-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Missing keys: []
Unexpected keys: []


Convert to Onnx

In [ ]:
!pip install onnx onnxscript
!pip install optimum[onnxruntime]
from optimum.onnxruntime import ORTModelForImageClassification

save_dir = "/content/drive/MyDrive/Thesis/models/swin"
model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)

onnx_model = ORTModelForImageClassification.from_pretrained(
    save_dir,
    export=True,
)

onnx_model.save_pretrained("/content/drive/MyDrive/Thesis/models/swin_onnx")

Quantize ONNX

In [ ]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

onnx_dir = "/content/drive/MyDrive/Thesis/models/swin_onnx"
quantized_dir = "/content/drive/MyDrive/Thesis/models/swin_onnx_int8"

quantizer = ORTQuantizer.from_pretrained(onnx_dir)

qconfig = AutoQuantizationConfig.arm64(
    is_static=False,
    per_channel=False,
)

quantizer.quantize(
    save_dir=quantized_dir,
    quantization_config=qconfig,
)

  elem_type: 7
  shape {
    dim {
      dim_param: "unk__12"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__76"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__578"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__641"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__1143"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__1206"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__1694"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__1757"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__2245"
    }
    dim {
      dim_value: 2
   

PosixPath('/content/drive/MyDrive/Thesis/models/swin_onnx_int8')